## Search
From Poole and Mackworth

In [ ]:
# display.py - A simple way to trace the intermediate steps of algorithms.

class Displayable(object):
    """Class that uses 'display'.
    The amount of detail is controlled by max_display_level
    """
    max_display_level = 4   # can be overridden in subclasses or instances

    def display(self,level,*args,**nargs):
        """print the arguments if level is less than or equal to the
        current max_display_level.
        level is an integer.
        the other arguments are whatever arguments print can take.
        """
        if level <= self.max_display_level:
            print(*args, **nargs)  ##if error you are using Python2 not Python3



# Defining a Search Problem

In [ ]:
import matplotlib.pyplot as plt
import random

class Search_problem(Displayable):
    """A search problem consists of:
    * a start node
    * a neighbors function that gives the neighbors of a node
    * a specification of a goal
    * a (optional) heuristic function.
    The methods must be overridden to define a search problem."""

    def start_node(self):
        """returns start node"""
        raise NotImplementedError("start_node")   # abstract method
    
    def is_goal(self,node):
        """is True if node is a goal"""
        raise NotImplementedError("is_goal")   # abstract method

    def neighbors(self,node):
        """returns a list (or enumeration) of the arcs for the neighbors of node"""
        raise NotImplementedError("neighbors")   # abstract method

    def heuristic(self,n):
        """Gives the heuristic value of node n.
        Returns 0 if not overridden."""
        return 0



# Defining an Arc

In [ ]:
class Arc(object):
    """An arc consists of 
       a from_node and a to_node node 
       a (non-negative) cost
       an (optional) action
    """
    def __init__(self, from_node, to_node, cost=1, action=None):
        self.from_node = from_node
        self.to_node = to_node
        self.cost = cost
        assert cost >= 0, (f"Cost cannot be negative: {self}, cost={cost}")
        self.action = action

    def __repr__(self):
        """string representation of an arc"""
        if self.action:
            return f"{self.from_node} --{self.action}--> {self.to_node}"
        else:
            return f"{self.from_node} --> {self.to_node}"



# Defining a search problem from explicit graph

In [ ]:
class Search_problem_from_explicit_graph(Search_problem):
    """A search problem from an explicit graph.
    """

    def __init__(self, title, nodes, arcs, start=None, goals=set(), hmap={},
                     positions=None):
        """ A search problem consists of:
        * list or set of nodes
        * list or set of arcs
        * start node
        * list or set of goal nodes
        * hmap: dictionary that maps each node into its heuristic value.
        * positions: dictionary that maps each node into its (x,y) position
        """
        self.title = title
        self.neighs = {}
        self.nodes = nodes
        for node in nodes:
            self.neighs[node]=[]
        self.arcs = arcs
        for arc in arcs:
            self.neighs[arc.from_node].append(arc)
        self.start = start
        self.goals = goals
        self.hmap = hmap
        if positions is None:
            self.positions = {node:(random.random(),random.random()) for node in nodes}
        else:
            self.positions = positions

    def start_node(self):
        """returns start node"""
        return self.start
    
    def is_goal(self,node):
        """is True if node is a goal"""
        return node in self.goals

    def neighbors(self,node):
        """returns the neighbors of node (a list of arcs)"""
        return self.neighs[node]

    def heuristic(self,node):
        """Gives the heuristic value of node n.
        Returns 0 if not overridden in the hmap."""
        if node in self.hmap:
            return self.hmap[node]
        else:
            return 0
        
    def __repr__(self):
        """returns a string representation of the search problem"""
        res=""
        for arc in self.arcs:
            res += f"{arc}.  "
        return res

    def show(self, fontsize=10, node_color='orange', show_costs = True):
        """Show the graph as a figure
        """
        self.fontsize = fontsize
        self.show_costs = show_costs
        plt.ion()   # interactive
        fig, ax = plt.subplots()
        ax.set_axis_off()
        ax.set_title(self.title, fontsize=fontsize)
        self.show_graph(ax, node_color)

    def show_graph(self, ax, node_color='orange'): 
        bbox = dict(boxstyle="round4,pad=1.0,rounding_size=0.5",facecolor=node_color)
        for arc in self.arcs:
            self.show_arc(ax, arc)
        for node in self.nodes:
            self.show_node(ax, node, node_color = node_color)

    def show_node(self, ax, node, node_color):
            x,y = self.positions[node]
            ax.text(x,y,node,bbox=dict(boxstyle="round4,pad=1.0,rounding_size=0.5",
                                    facecolor=node_color),
                        ha='center',va='center', fontsize=self.fontsize)
        
    def show_arc(self, ax, arc, arc_color='black', node_color='white'):
            from_pos = self.positions[arc.from_node]
            to_pos = self.positions[arc.to_node]
            ax.annotate(arc.to_node, from_pos, xytext=to_pos,
                        arrowprops={'arrowstyle':'<|-', 'linewidth': 2,
                                            'color':arc_color},
                        bbox=dict(boxstyle="round4,pad=1.0,rounding_size=0.5",
                                                 facecolor=node_color),
                                ha='center',va='center',
                                fontsize=self.fontsize)
            # Add costs to middle of arcs:
            if self.show_costs:
                ax.text((from_pos[0]+to_pos[0])/2, (from_pos[1]+to_pos[1])/2,
                         arc.cost, bbox=dict(pad=1,fc='w',ec='w'),
                         ha='center',va='center',fontsize=self.fontsize)



# Defining a path

In [ ]:

class Path(object):
    """A path is either a node or a path followed by an arc"""
    
    def __init__(self,initial,arc=None):
        """initial is either a node (in which case arc is None) or
        a path (in which case arc is an object of type Arc)"""
        self.initial = initial
        self.arc=arc
        if arc is None:
            self.cost=0
        else:
            self.cost = initial.cost+arc.cost

    def end(self):
        """returns the node at the end of the path"""
        if self.arc is None:
            return self.initial
        else:
            return self.arc.to_node

    def nodes(self):
        """enumerates the nodes of the path from the last element backwards
        """
        current = self
        while current.arc is not None:
            yield current.arc.to_node
            current = current.initial
        yield current.initial

    def initial_nodes(self):
        """enumerates the nodes for the path before the end node.
        This calls nodes() for the initial part of the path.
        """
        if self.arc is not None:
            yield from self.initial.nodes()

    def __repr__(self):
        """returns a string representation of a path"""
        if self.arc is None:
            return str(self.initial)
        elif self.arc.action:
            return f"{self.initial}\n   --{self.arc.action}--> {self.arc.to_node}"
        else:
            return f"{self.initial} --> {self.arc.to_node}"



# Search Examples

In [ ]:
# searchExample.py - Search Examples
# AIFCA Python code Version 0.9.18 Documentation at https://aipython.org
# Download the zip file and read aipython.pdf for documentation

# Artificial Intelligence: Foundations of Computational Agents https://artint.info
# Copyright 2017-2026 David L. Poole and Alan K. Mackworth
# This work is licensed under a Creative Commons
# Attribution-NonCommercial-ShareAlike 4.0 International License.
# See: https://creativecommons.org/licenses/by-nc-sa/4.0/deed.en

#from searchProblem import Arc, Search_problem_from_explicit_graph, Search_problem

problem1 = Search_problem_from_explicit_graph('Problem 1',
    {'A','B','C','D','G'},
    [Arc('A','B',3), Arc('A','C',1), Arc('B','D',1), Arc('B','G',3),
         Arc('C','B',1), Arc('C','D',3), Arc('D','G',1)],
    start = 'A',
    goals = {'G'},
    positions={'A': (0, 1), 'B': (0.5, 0.5), 'C': (0,0.5),
                   'D': (0.5,0), 'G': (1,0)})

problem2 = Search_problem_from_explicit_graph('Problem 2',
    {'A','B','C','D','E','G','H','J'},
    [Arc('A','B',1), Arc('B','C',3), Arc('B','D',1), Arc('D','E',3),
        Arc('D','G',1), Arc('A','H',3), Arc('H','J',1)],
    start = 'A',
    goals = {'G'},
    positions={'A':(0, 1), 'B':(0, 3/4), 'C':(0,0), 'D':(1/4,3/4),
                   'E':(1/4,0), 'G':(2/4,3/4), 'H':(3/4,1), 'J':(3/4,3/4)})

problem3 = Search_problem_from_explicit_graph('Problem 3',
    {'a','b','c','d','e','g','h','j'},
    [],
    start = 'g',
    goals = {'k','g'})

simp_delivery_graph = Search_problem_from_explicit_graph("Acyclic Delivery Graph",
    {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J'},
    [    Arc('A', 'B', 2),
         Arc('A', 'C', 3),
         Arc('A', 'D', 4),
         Arc('B', 'E', 2),
         Arc('B', 'F', 3),
         Arc('C', 'J', 7),
         Arc('D', 'H', 4),
         Arc('F', 'D', 2),
         Arc('H', 'G', 3),
         Arc('J', 'G', 4)],
   start = 'A',
   goals = {'G'},
   hmap = {
        'A': 7,
        'B': 5,
        'C': 9,
        'D': 6,
        'E': 3,
        'F': 5,
        'G': 0,
        'H': 3,
        'J': 4,
    },
    positions = {
        'A': (0.4,0.1),
        'B': (0.4,0.4),
        'C': (0.1,0.1),
        'D': (0.7,0.1),
        'E': (0.6,0.7),
        'F': (0.7,0.4),
        'G': (0.7,0.9),
        'H': (0.9,0.6),
        'J': (0.3,0.9)
        }
    )
cyclic_simp_delivery_graph = Search_problem_from_explicit_graph("Cyclic Delivery Graph",
    {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J'},
    [    Arc('A', 'B', 2),
         Arc('A', 'C', 3),
         Arc('A', 'D', 4),
         Arc('B', 'E', 2),
         Arc('B', 'F', 3),
         Arc('C', 'A', 3),
         Arc('C', 'J', 6),
         Arc('D', 'A', 4),
         Arc('D', 'H', 4),
         Arc('F', 'B', 3),
         Arc('F', 'D', 2),
         Arc('G', 'H', 3),
         Arc('G', 'J', 4),
         Arc('H', 'D', 4),
         Arc('H', 'G', 3),
         Arc('J', 'C', 6),
         Arc('J', 'G', 4)],
   start = 'A',
   goals = {'G'},
   hmap = {
        'A': 7,
        'B': 5,
        'C': 9,
        'D': 6,
        'E': 3,
        'F': 5,
        'G': 0,
        'H': 3,
        'J': 4,
    },
    positions = {
        'A': (0.4,0.1),
        'B': (0.4,0.4),
        'C': (0.1,0.1),
        'D': (0.7,0.1),
        'E': (0.6,0.7),
        'F': (0.7,0.4),
        'G': (0.7,0.9),
        'H': (0.9,0.6),
        'J': (0.3,0.9)
        })

tree_graph = Search_problem_from_explicit_graph("Tree Graph",
    {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O',
         'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'AA', 'BB', 'CC',
         'DD', 'EE', 'FF', 'GG', 'HH', 'II', 'JJ', 'KK'},
    [    Arc('A', 'B', 1),
         Arc('A', 'C', 1),
         Arc('B', 'D', 1),
         Arc('B', 'E', 1),
         Arc('C', 'F', 1),
         Arc('C', 'G', 1),
         Arc('D', 'H', 1),
         Arc('D', 'I', 1),
         Arc('E', 'J', 1),
         Arc('E', 'K', 1),
         Arc('F', 'L', 1),
         Arc('G', 'M', 1),
         Arc('G', 'N', 1),
         Arc('H', 'O', 1),
         Arc('H', 'P', 1),
         Arc('J', 'Q', 1),
         Arc('J', 'R', 1),
         Arc('L', 'S', 1),
         Arc('L', 'T', 1),
         Arc('N', 'U', 1),
         Arc('N', 'V', 1),
         Arc('O', 'W', 1),
         Arc('P', 'X', 1),
         Arc('P', 'Y', 1),
         Arc('R', 'Z', 1),
         Arc('R', 'AA', 1),
         Arc('T', 'BB', 1),
         Arc('T', 'CC', 1),
         Arc('V', 'DD', 1),
         Arc('V', 'EE', 1),
         Arc('W', 'FF', 1),
         Arc('X', 'GG', 1),
         Arc('Y', 'HH', 1),
         Arc('AA', 'II', 1),
         Arc('CC', 'JJ', 1),
         Arc('CC', 'KK', 1)
    ],
   start = 'A',
   goals = {'K', 'M', 'T', 'X', 'Z', 'HH'},
    positions = {
        'A': (0.5,0.95),
        'B': (0.3,0.8),
        'C': (0.7,0.8),
        'D': (0.2,0.65),
        'E': (0.4,0.65),
        'F': (0.6,0.65),
        'G': (0.8,0.65),
        'H': (0.2,0.5),
        'I': (0.3,0.5),
        'J': (0.4,0.5),
        'K': (0.5,0.5),
        'L': (0.6,0.5),
        'M': (0.7,0.5),
        'N': (0.8,0.5),
        'O': (0.1,0.35),
        'P': (0.2,0.35),
        'Q': (0.3,0.35),
        'R': (0.4,0.35),
        'S': (0.5,0.35),
        'T': (0.6,0.35),
        'U': (0.7,0.35),
        'V': (0.8,0.35),
        'W': (0.1,0.2),
        'X': (0.2,0.2),
        'Y': (0.3,0.2),
        'Z': (0.4,0.2),
        'AA': (0.5,0.2),
        'BB': (0.6,0.2),
        'CC': (0.7,0.2),
        'DD': (0.8,0.2),
        'EE': (0.9,0.2),
        'FF': (0.1,0.05),
        'GG': (0.2,0.05),
        'HH': (0.3,0.05),
        'II': (0.5,0.05),
        'JJ': (0.7,0.05),
        'KK': (0.8,0.05)      
        }
    )

# tree_graph.show(show_costs = False)


# The map of Romania

In [ ]:
romania_map = Search_problem_from_explicit_graph("Romania",
    {'Arad', 'Bucharest', 'Craiova', 'Dobreta', 'Eforie', 'Fagaras', 'Giurgiu', 'Hirsova',
     'Iasi', 'Lugoj', 'Mehadia', 'Neamt', 'Oradea', 'Pitesti', 'Rimnicu Vilcea', 'Sibiu',
     'Timisoara', 'Urziceni', 'Vaslui', 'Zerind'},
    [
        Arc('Arad', 'Zerind', 75),  Arc('Zerind', 'Arad', 75),
        Arc('Arad', 'Sibiu', 140),  Arc('Sibiu', 'Arad', 140),
        Arc('Arad', 'Timisoara', 118),  Arc('Timisoara', 'Arad', 118),
        Arc('Zerind', 'Oradea', 71),  Arc('Oradea', 'Zerind', 71),
        Arc('Oradea', 'Sibiu', 151),  Arc('Sibiu', 'Oradea', 151),
        Arc('Timisoara', 'Lugoj', 111),  Arc('Lugoj', 'Timisoara', 111),
        Arc('Lugoj', 'Mehadia', 70),  Arc('Mehadia', 'Lugoj', 70),
        Arc('Mehadia', 'Dobreta', 75),  Arc('Dobreta', 'Mehadia', 75),
        Arc('Dobreta', 'Craiova', 120),  Arc('Craiova', 'Dobreta', 120),
        Arc('Craiova', 'Rimnicu Vilcea', 146),  Arc('Rimnicu Vilcea', 'Craiova', 146),
        Arc('Craiova', 'Pitesti', 138),  Arc('Pitesti', 'Craiova', 138),
        Arc('Sibiu', 'Fagaras', 99),  Arc('Fagaras', 'Sibiu', 99),
        Arc('Sibiu', 'Rimnicu Vilcea', 80),  Arc('Rimnicu Vilcea', 'Sibiu', 80),
        Arc('Rimnicu Vilcea', 'Pitesti', 97),  Arc('Pitesti', 'Rimnicu Vilcea', 97),
        Arc('Fagaras', 'Bucharest', 211),  Arc('Bucharest', 'Fagaras', 211),
        Arc('Pitesti', 'Bucharest', 101),  Arc('Bucharest', 'Pitesti', 101),
        Arc('Bucharest', 'Giurgiu', 90),  Arc('Giurgiu', 'Bucharest', 90),
        Arc('Bucharest', 'Urziceni', 85),  Arc('Urziceni', 'Bucharest', 85),
        Arc('Urziceni', 'Hirsova', 98),  Arc('Hirsova', 'Urziceni', 98),
        Arc('Hirsova', 'Eforie', 86),  Arc('Eforie', 'Hirsova', 86),
        Arc('Urziceni', 'Vaslui', 142),  Arc('Vaslui', 'Urziceni', 142),
        Arc('Vaslui', 'Iasi', 92),  Arc('Iasi', 'Vaslui', 92),
        Arc('Iasi', 'Neamt', 87),  Arc('Neamt', 'Iasi', 87)
    ],
    start = 'Arad',
    goals = {'Bucharest'},
    hmap = {
        'Arad': 366, 'Bucharest': 0, 'Craiova': 160, 'Dobreta': 242, 'Eforie': 161,
        'Fagaras': 176, 'Giurgiu': 77, 'Hirsova': 151, 'Iasi': 226, 'Lugoj': 244,
        'Mehadia': 241, 'Neamt': 234, 'Oradea': 380, 'Pitesti': 100,
        'Rimnicu Vilcea': 193, 'Sibiu': 253, 'Timisoara': 329, 'Urziceni': 80,
        'Vaslui': 199, 'Zerind': 374,
    },
    positions = {
        'Arad': (0.01, 0.72), 'Bucharest': (0.64, 0.2), 'Craiova': (0.34, 0.08),
        'Dobreta': (0.16, 0.11), 'Eforie': (0.97, 0.1), 'Fagaras': (0.45, 0.59),
        'Giurgiu': (0.59, 0.02), 'Hirsova': (0.91, 0.27), 'Iasi': (0.79, 0.77),
        'Lugoj': (0.16, 0.36), 'Mehadia': (0.17, 0.24), 'Neamt': (0.65, 0.86),
        'Oradea': (0.09, 0.98), 'Pitesti': (0.48, 0.33), 'Rimnicu Vilcea': (0.3, 0.46),
        'Sibiu': (0.25, 0.61), 'Timisoara': (0.02, 0.46), 'Urziceni': (0.76, 0.27),
        'Vaslui': (0.86, 0.57), 'Zerind': (0.05, 0.85)
        })

# Generic Searcher

In [ ]:


class Searcher(Displayable):
    """returns a searcher for a problem.
    Paths can be found by repeatedly calling search().
    This does depth-first search unless overridden
    """
    def __init__(self, problem):
        """creates a searcher from a problem
        """
        self.problem = problem
        self.initialize_frontier()
        self.num_expanded = 0
        self.add_to_frontier(Path(problem.start_node()))
        super().__init__()

    def initialize_frontier(self):
        self.frontier = []
        
    def empty_frontier(self):
        return self.frontier == []
        
    def add_to_frontier(self,path):
        self.frontier.append(path)
        
    def search(self):
        """returns (next) path from the problem's start node
        to a goal node. 
        Returns None if no path exists.
        """
        while not self.empty_frontier():
            #
            self.path = self.frontier.pop()
            #
            self.num_expanded += 1
            if self.problem.is_goal(self.path.end()):    # solution found
                self.solution = self.path   # store the solution found
                self.display(1, f"Solution: {self.path} (cost: {self.path.cost})\n",
                    self.num_expanded, "paths have been expanded and",
                            len(self.frontier), "paths remain in the frontier")
                return self.path
            else:
                self.display(4,f"Expanding: {self.path} (cost: {self.path.cost})")
                neighs = self.problem.neighbors(self.path.end())
                self.display(2,f"Expanding: {self.path} with neighbors {neighs}")
                for arc in reversed(list(neighs)):
                    self.add_to_frontier(Path(self.path,arc))
                self.display(3, f"New frontier: {[p.end() for p in self.frontier]}")

        self.display(0,"No (more) solutions. Total of",
                     self.num_expanded,"paths expanded.")



    


# Specialized Searchers

In [ ]:

import heapq        # part of the Python standard library


class FrontierPQ(object):
    """A frontier consists of a priority queue (heap), frontierpq, of
        (value, index, path) triples, where
    * value is the value we want to minimize (e.g., path cost + h).
    * index is a unique index for each element
    * path is the path on the queue
    Note that the priority queue always returns the smallest element.
    """

    def __init__(self):
        """constructs the frontier, initially an empty priority queue 
        """
        self.frontier_index = 0  # the number of items added to the frontier
        self.frontierpq = []  # the frontier priority queue

    def empty(self):
        """is True if the priority queue is empty"""
        return self.frontierpq == []

    def add(self, path, value):
        """add a path to the priority queue
        value is the value to be minimized"""
        self.frontier_index += 1    # get a new unique index
        heapq.heappush(self.frontierpq,(value, -self.frontier_index, path))

    def pop(self):
        """returns and removes the path of the frontier with minimum value.
        """
        (_,_,path) = heapq.heappop(self.frontierpq)
        return path 

    def count(self,val):
        """returns the number of elements of the frontier with value=val"""
        return sum(1 for e in self.frontierpq if e[0]==val)

    def __repr__(self):
        """string representation of the frontier"""
        return str([(n,c,str(p)) for (n,c,p) in self.frontierpq])
    
    def __len__(self):
        """length of the frontier"""
        return len(self.frontierpq)

    def __iter__(self):
        """iterate through the paths in the frontier"""
        for (_,_,path) in self.frontierpq:
            yield path
    
class AStarSearcher(Searcher2):
    """returns a searcher for a problem.
    Paths can be found by repeatedly calling search().
    """

    def __init__(self, problem):
        super().__init__(problem)

    def initialize_frontier(self):
        self.frontier = FrontierPQ()

    def empty_frontier(self):
        return self.frontier.empty()

    def add_to_frontier(self,path):
        """add path to the frontier with the appropriate cost"""
        value = path.cost+self.problem.heuristic(path.end())
        self.frontier.add(path, value)



In [ ]:
searcher1 = Searcher(simp_delivery_graph)

In [ ]:
searcher1.search()

In [ ]:
searcher2 = AStarSearcher(simp_delivery_graph)

In [ ]:
searcher2.search()

In [ ]:
searcher2.search()

In [ ]:
searcher2.search()

In [ ]:
searcher2.search()

# A*

In [ ]:
searcher3  = AStarSearcher(cyclic_simp_delivery_graph)

In [ ]:
searcher3.search()

In [ ]:
searcher3.search()

In [ ]:
#can continue

# Multiple Path Pruning

In [ ]:
class SearcherMPP(AStarSearcher):
    """returns a searcher for a problem.
    Paths can be found by repeatedly calling search().
    """
    def __init__(self, problem):
        super().__init__(problem)
        self.explored = set()

    def search(self):
        """returns next path from an element of problem's start nodes
        to a goal node. 
        Returns None if no path exists.
        """
        while not self.empty_frontier():
            self.path = self.frontier.pop()
            if self.path.end() not in self.explored:
                if not(self.problem.is_goal(self.path.end())):
                   self.explored.add(self.path.end())
                self.num_expanded += 1
                if self.problem.is_goal(self.path.end()):
                    self.solution = self.path   # store the solution found
                    self.display(1, f"Solution: {self.path} (cost: {self.path.cost})\n",
                    self.num_expanded, "paths have been expanded and",
                            len(self.frontier), "paths remain in the frontier")
                    return self.path
                else:
                    self.display(4,f"Expanding: {self.path} (cost: {self.path.cost})")
                    neighs = self.problem.neighbors(self.path.end())
                    self.display(2,f"Expanding: {self.path} with neighbors {neighs}")
                    for arc in neighs:
                        self.add_to_frontier(Path(self.path,arc))
                    self.display(3, f"New frontier: {[p.end() for p in self.frontier]}")
        self.display(0,"No (more) solutions. Total of",
                     self.num_expanded,"paths expanded.")







In [ ]:
searcherMPP = SearcherMPP(cyclic_simp_delivery_graph)

In [ ]:
searcherMPP.search()

#  Your Experiments with the Romania Map go here